In [9]:
"""
Model 1: SVM classifier for term-deposit subscription prediction
Dataset: UCI Bank Marketing, bank-additional-full variant
(archive.ics.uci.edu/dataset/222/bank+marketing)

WHAT THIS SCRIPT DOES
----------------------
1. Loads the raw client/campaign-level Bank Marketing data (semicolon-
   separated, 41,188 rows, direct-marketing calls made by a Portuguese
   bank to sell a term deposit product).
2. Cleans it (strips quote characters left over from the source CSV).
3. Builds a binary label: did the client subscribe to a term deposit
   ("y" == "yes")? Positive class is a 11.3% minority.
4. Selects a deliberately limited, PRE-CALL feature set: client profile
   (age, job) plus prior campaign contact history (contact channel,
   number of contacts this campaign, days since last contact, number of
   previous contacts, outcome of the previous campaign). The "duration"
   attribute (length of the call in seconds) is intentionally EXCLUDED:
   the dataset's own UCI documentation states duration is not known
   before a call is placed and "should be discarded if the intention is
   to have a realistic predictive model" (it leaks the outcome, since a
   long call is itself evidence of interest).
5. One-hot encodes the categorical features, scales, trains an SVM
   (RBF kernel, class_weight="balanced" to counter the 11.3% skew).
6. Reports accuracy, precision, recall, F1 and a confusion matrix.

REPRODUCIBILITY NOTE ON SAMPLE SIZE
------------------------------------
RBF-kernel SVMs scale roughly quadratically with training-set size.
Training on all ~41k rows was impractically slow in the sandbox used to
produce this report, so a stratified random sample of 6,000 clients
(preserving the 11.3% subscription rate) was drawn before the
train/test split. This is documented here and in the report appendix.

HOW TO USE
----------
Download the dataset from:
  https://archive.ics.uci.edu/dataset/222/bank+marketing
It comes as "bank-additional-full.csv". Place it in a `data/` folder
next to this script, or edit DATA_PATH below.

NOTE ON INTERPRETATION
-----------------------
The "Interpretation" block printed at the end summarises the same reading
of these numbers used in the report's Insights section (Section 3), so the
script's output is self-explanatory on its own. It is a starting point,
not a substitute for your own analysis -- the full discussion, evaluation-
metric justification, and business framing in Part 1.3 still have to be
your own argument (Condition 3 AI policy). Treat the printed numbers as
the evidence, and the interpretation as a claim you should be able to
defend yourself if asked.
"""

# Model 1: SVM classifier for term-deposit subscription prediction
# Dataset: UCI Bank Marketing (archive.ics.uci.edu/dataset/222/bank+marketing)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

RANDOM_STATE = 42
DATA_PATH = "data/bank-additional-full.csv"
SAMPLE_SIZE = 6000


In [10]:
# Load and clean (semicolon-separated, strip stray quote characters)
df = pd.read_csv(DATA_PATH, sep=";")
df.columns = [c.strip(chr(34)) for c in df.columns]
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].str.strip(chr(34))
df["Subscribed"] = (df["y"] == "yes").astype(int)
df.head()


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,Subscribed
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,0
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,0
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,0
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,0
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,0


In [11]:
# Stratified sample for tractable RBF-SVM training time (~41k rows is too slow for RBF SVM)
df_sample, _ = train_test_split(
    df, train_size=SAMPLE_SIZE, random_state=RANDOM_STATE, stratify=df["Subscribed"]
)
print(f"Clients (full dataset): {len(df)}")
print(f"Subscribed (full dataset): {df['Subscribed'].sum()} ({df['Subscribed'].mean()*100:.1f}%)")
print(f"Stratified sample used for modelling: {len(df_sample)}")
print(f"Subscribed (sample): {df_sample['Subscribed'].sum()} ({df_sample['Subscribed'].mean()*100:.1f}%)")
df_sample.head()


Clients (full dataset): 41188
Subscribed (full dataset): 4640 (11.3%)
Stratified sample used for modelling: 6000
Subscribed (sample): 676 (11.3%)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,Subscribed
27661,42,admin.,divorced,university.degree,no,yes,no,cellular,nov,fri,...,999,0,nonexistent,-0.1,93.200,-42.0,4.021,5195.8,no,0
33262,27,services,married,high.school,no,yes,no,cellular,may,tue,...,999,1,failure,-1.8,92.893,-46.2,1.291,5099.1,no,0
40174,32,admin.,single,university.degree,no,no,no,cellular,jul,fri,...,9,3,failure,-1.7,94.215,-40.3,0.861,4991.6,no,0
31391,33,services,married,high.school,no,yes,yes,cellular,may,wed,...,999,1,failure,-1.8,92.893,-46.2,1.334,5099.1,no,0
16981,59,retired,single,professional.course,unknown,yes,no,cellular,jul,thu,...,999,0,nonexistent,1.4,93.918,-42.7,4.962,5228.1,no,0


In [12]:
# Feature set: pre-call client profile + campaign contact history.
# "duration" is intentionally EXCLUDED -- per the UCI documentation it is not known
# before a call is placed and leaks the outcome (a long call almost always means interest).
NUMERIC_FEATURES = ["age", "campaign", "pdays", "previous"]
CATEGORICAL_FEATURES = ["job", "contact", "poutcome"]

X = df_sample[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df_sample["Subscribed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

preprocess = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
])


In [13]:
# Fit SVM (RBF kernel, class_weight="balanced" to counter the 11.3% skew) and evaluate
model = Pipeline([
    ("prep", preprocess),
    ("svm", SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", random_state=RANDOM_STATE)),
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("--- Evaluation metrics ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1-score:  {f1_score(y_test, y_pred):.3f}")


--- Evaluation metrics ---
Accuracy:  0.721
Precision: 0.207
Recall:    0.521
F1-score:  0.296


In [14]:
# Confusion matrix and full classification report
print("Confusion matrix (rows=actual, cols=predicted):")
cm = confusion_matrix(y_test, y_pred)
print(cm)
print()
print(classification_report(y_test, y_pred))

# --- Interpretation (see NOTE ON INTERPRETATION in the .py version) ---
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
caught, missed = cm[1, 1], cm[1, 0]
poutcome_rate = df_sample.groupby("poutcome")["Subscribed"].mean()

print("\n--- Interpretation ---")
print(
    f"With duration withheld and class weights balanced against the "
    f"{df_sample['Subscribed'].mean()*100:.1f}% skew, the SVM recovers moderate recall "
    f"({rec*100:.1f}%, {caught} of {caught + missed} subscribers caught) at the cost of "
    f"precision ({prec*100:.1f}%) -- trading false negatives for false positives. That is a "
    f"defensible choice for a lead-prioritisation list (contact more people to avoid missing "
    f"a genuine prospect), not for a precision-targeted campaign."
)
if "success" in poutcome_rate.index and "nonexistent" in poutcome_rate.index:
    print(
        f"Prior campaign outcome remains the strongest legitimate (non-leaky) signal: clients "
        f"whose previous campaign succeeded subscribe {poutcome_rate['success']*100:.1f}% of "
        f"the time, versus {poutcome_rate['nonexistent']*100:.1f}% for clients never previously "
        f"contacted."
    )


Confusion matrix (rows=actual, cols=predicted):
[[994 337]
 [ 81  88]]

              precision    recall  f1-score   support

           0       0.92      0.75      0.83      1331
           1       0.21      0.52      0.30       169

    accuracy                           0.72      1500
   macro avg       0.57      0.63      0.56      1500
weighted avg       0.84      0.72      0.77      1500


--- Interpretation ---
With duration withheld and class weights balanced against the 11.3% skew, the SVM recovers moderate recall (52.1%, 88 of 169 subscribers caught) at the cost of precision (20.7%) -- trading false negatives for false positives. That is a defensible choice for a lead-prioritisation list (contact more people to avoid missing a genuine prospect), not for a precision-targeted campaign.
Prior campaign outcome remains the strongest legitimate (non-leaky) signal: clients whose previous campaign succeeded subscribe 67.5% of the time, versus 8.9% for clients never previously contac

In [15]:
df_sample.to_csv("bank_marketing_sample.csv", index=False)
print("Saved bank_marketing_sample.csv")


Saved bank_marketing_sample.csv


In [16]:
# Figures: confusion matrix heatmap + Age/Campaign scatter
import numpy as np
import matplotlib
matplotlib.use("Agg")  # comment this out if running interactively and you want inline plots
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams.update({"font.size": 10})

cm = confusion_matrix(y_test, y_pred)  # (already computed earlier; recomputed here for a self-contained cell)

# --- Confusion matrix heatmap ---
fig, ax = plt.subplots(figsize=(3.2, 2.8))
im = ax.imshow(cm, cmap="Blues")
labels = ["No", "Subscribed"]
ax.set_xticks([0, 1]); ax.set_xticklabels(labels)
ax.set_yticks([0, 1]); ax.set_yticklabels(labels)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=11)
ax.set_title("Model 1: SVM confusion matrix", fontsize=10)
fig.tight_layout()
fig.savefig("fig_svm_confusion_matrix.png", dpi=200)
plt.show()

# --- Age/Campaign scatter, coloured by actual class, misclassified ringed ---
correct = (y_test.values == y_pred)
colors = np.where(y_test.values == 1, "tab:orange", "tab:blue")
age_t = X_test["age"].values
camp_t = X_test["campaign"].values

fig, ax = plt.subplots(figsize=(3.6, 2.8))
ax.scatter(age_t[correct], camp_t[correct], c=colors[correct], alpha=0.5, s=14)
ax.scatter(age_t[~correct], camp_t[~correct],
           facecolors="none", edgecolors="red", s=30, linewidths=1.0)
ax.set_xlabel("Age (years)")
ax.set_ylabel("Campaign (contacts this campaign)")
ax.set_ylim(0, np.percentile(camp_t, 98))
ax.set_title("Model 1: Age/Campaign by actual class", fontsize=10)
legend_elems = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='tab:orange', markersize=6, label='Subscribed (actual)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='tab:blue', markersize=6, label='No (actual)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='none', markeredgecolor='red', markersize=7, label='Misclassified'),
]
ax.legend(handles=legend_elems, fontsize=6.5, loc="upper right")
fig.tight_layout()
fig.savefig("fig_svm_age_campaign_scatter.png", dpi=200)
plt.show()

print("Saved fig_svm_confusion_matrix.png and fig_svm_age_campaign_scatter.png")


Saved fig_svm_confusion_matrix.png and fig_svm_age_campaign_scatter.png


C:\Users\Shubh\AppData\Local\Temp\ipykernel_24928\40589291.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Shubh\AppData\Local\Temp\ipykernel_24928\40589291.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
